# Pipeline de ETL dos dados do Censo Escolar

Este projeto busca unificar todo o processo de etl para consumo pelos panoramas do Observatório Curica.

Os dados do Censo Escolar a partir do ano de 2019 foram baixados para o diretório do projeto e os arquivos do dicionário de dados e os arquivos com os dados em si foram descompactados, renomeados e copiados para os diretórios respctivos. Esses arquivos não são versionados no GitHub em razão de seu tamanho.

A partir do ano de 2025, o questionário do censo escolar foi bastante ampliado e os dados estão divididos em diferentes tabelas.

## Primeira transformação - Bronze

In [1]:
import gc
import pandas as pd
from pathlib import Path

from schema import SCHEMA_CURICA


In [2]:
# toma como diretório pai o diretório do arquivo do notebook
BASE_DIR = Path("etl_censo.ipynb").resolve().parent
RAW_CENSO_DIR = BASE_DIR / "data" / "raw"

In [3]:
# Procurar todos os arquivos CSV dentro de data/raw/censo 
arquivos_censo = sorted(RAW_CENSO_DIR.glob("*.csv"))

if not arquivos_censo:
    print("⚠️ Nenhum arquivo CSV encontrado em:", RAW_CENSO_DIR)
else:
    print(f"✅ {len(arquivos_censo)} arquivos encontrados:\n")
    for arq in arquivos_censo:
        print("-", arq.name)

✅ 10 arquivos encontrados:

- Tabela_Docente_2025.csv
- Tabela_Escola_2025.csv
- Tabela_Matricula_2025.csv
- Tabela_Turma_2025.csv
- microdados_ed_basica_2019.csv
- microdados_ed_basica_2020.csv
- microdados_ed_basica_2021.csv
- microdados_ed_basica_2022.csv
- microdados_ed_basica_2023.csv
- microdados_ed_basica_2024.csv


In [4]:
# carregar tabelas
df_censo_2019 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2019.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2019 = df_censo_2019[df_censo_2019["SG_UF"] == "AC"].copy()
del df_censo_2019
gc.collect()

df_censo_2020 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2020.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2020 = df_censo_2020[df_censo_2020["SG_UF"] == "AC"].copy()
del df_censo_2020
gc.collect()

df_censo_2021 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2021.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2021 = df_censo_2021[df_censo_2021["SG_UF"] == "AC"].copy()
del df_censo_2021
gc.collect()

df_censo_2022 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2022.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2022 = df_censo_2022[df_censo_2022["SG_UF"] == "AC"].copy()
del df_censo_2022
gc.collect()

df_censo_2023 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2023.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2023 = df_censo_2023[df_censo_2023["SG_UF"] == "AC"].copy()
del df_censo_2023
gc.collect()

df_censo_2024 = pd.read_csv(RAW_CENSO_DIR / "microdados_ed_basica_2024.csv", sep=";", encoding="latin-1", low_memory=False)
df_censo_ac_2024 = df_censo_2024[df_censo_2024["SG_UF"] == "AC"].copy()
del df_censo_2024
gc.collect()

df_escola_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Escola_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_escola_ac_2025 = df_escola_2025[df_escola_2025["SG_UF"] == "AC"].copy()
del df_escola_2025
gc.collect()

df_matricula_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Matricula_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_matricula_ac_2025 = df_matricula_2025[df_matricula_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
del df_matricula_2025
gc.collect()

df_turma_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Turma_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_turma_ac_2025 = df_turma_2025[df_turma_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
del df_turma_2025
gc.collect()

df_docente_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Docente_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_docente_ac_2025 =  df_docente_2025[df_docente_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
del df_docente_2025
gc.collect()

0

In [5]:
print(df_censo_ac_2019.shape)
print(df_censo_ac_2020.shape)
print(df_censo_ac_2021.shape)
print(df_censo_ac_2022.shape)
print(df_censo_ac_2023.shape)
print(df_censo_ac_2024.shape)
print(df_escola_ac_2025.shape)
print(df_matricula_ac_2025.shape)
print(df_turma_ac_2025.shape)
print(df_docente_ac_2025.shape)


(1742, 370)
(1748, 370)
(1734, 370)
(1657, 385)
(1658, 408)
(1659, 426)
(1678, 302)
(1521, 237)
(1521, 190)
(1521, 156)


In [6]:
df = df_escola_ac_2025[df_escola_ac_2025['QT_MAT_BAS'] == 0]
df

KeyError: 'QT_MAT_BAS'

In [7]:
# filtra as escolas ativas
df_censo_ac_2019 = df_censo_ac_2019[df_censo_ac_2019["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_censo_ac_2020 = df_censo_ac_2020[df_censo_ac_2020["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_censo_ac_2021 = df_censo_ac_2021[df_censo_ac_2021["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_censo_ac_2022 = df_censo_ac_2022[df_censo_ac_2022["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_censo_ac_2023 = df_censo_ac_2023[df_censo_ac_2023["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_censo_ac_2024 = df_censo_ac_2024[df_censo_ac_2024["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()
df_escola_ac_2025 = df_escola_ac_2025[df_escola_ac_2025["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()

print("2019 - ", df_censo_ac_2019.shape)
print("2020 - ", df_censo_ac_2020.shape)
print("2021 - ", df_censo_ac_2021.shape)
print("2022 - ", df_censo_ac_2022.shape)
print("2023 - ", df_censo_ac_2023.shape)
print("2024 - ", df_censo_ac_2024.shape)
print("2025 - ", df_escola_ac_2025.shape)

2019 -  (1571, 370)
2020 -  (1565, 370)
2021 -  (1552, 370)
2022 -  (1530, 385)
2023 -  (1520, 408)
2024 -  (1523, 426)
2025 -  (1524, 302)


In [8]:
# Note que o número de escolas ativas na tabela de cadastro das escolas ainda é maior que o número de escolas na tabela de matrículas
# Para isso aqui é aplicado mais um filtro para selecionar apenas as escolas com matrículas em df_escola a partir da tabela de matrículas.
df_escola_ac_2025 = df_escola_ac_2025[df_escola_ac_2025["CO_ENTIDADE"].isin(df_matricula_ac_2025["CO_ENTIDADE"])].copy()

print("2025 - ", df_escola_ac_2025.shape)
print("2025 - ", df_matricula_ac_2025.shape)
print("2025 - ", df_turma_ac_2025.shape)
print("2025 - ", df_docente_ac_2025.shape)


2025 -  (1521, 302)
2025 -  (1521, 237)
2025 -  (1521, 190)
2025 -  (1521, 156)


## Merges - Prata

In [9]:
# merges do censo de 2025

df_censo_ac_2025 = df_escola_ac_2025.merge(
    df_matricula_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_mat")
)

df_censo_ac_2025.shape

(1521, 538)

In [10]:
df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_turma_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_tur")
)

df_censo_ac_2025.shape

(1521, 727)

In [11]:
df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_docente_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_doc")
)

df_censo_ac_2025.shape

(1521, 882)

## Definição das colunas de interesse antes do merge entre os diferentes anos

In [12]:
# comparação das colunas criadas entre os anos
def comparar_colunas(df1, df2, nome1="DF1", nome2="DF2"):

    col1 = set(df1.columns)
    col2 = set(df2.columns)

    somente_df1 = sorted(col1 - col2)
    somente_df2 = sorted(col2 - col1)

    print(f"\nColunas em {nome1} mas não em {nome2}:")
    print(f"Total: {len(somente_df1)}")
    print(somente_df1)

    print(f"\nColunas em {nome2} mas não em {nome1}:")
    print(f"Total: {len(somente_df2)}")
    print(somente_df2)



In [13]:

comparar_colunas(
    df_censo_ac_2024,
    df_censo_ac_2025,
    "Censo 2024",
    "Censo 2025"
)


Colunas em Censo 2024 mas não em Censo 2025:
Total: 41
['IN_DIURNO', 'IN_EAD', 'IN_EJA_FUND', 'IN_EJA_MED', 'IN_ESP', 'IN_ESP_CC', 'IN_ESP_CE', 'IN_FUND', 'IN_FUND_AF', 'IN_FUND_AI', 'IN_INF', 'IN_INF_CRE', 'IN_INF_PRE', 'IN_MED', 'IN_NOTURNO', 'IN_PROF', 'IN_PROF_ADMINISTRATIVOS', 'IN_PROF_AGRICOLA', 'IN_PROF_ALIMENTACAO', 'IN_PROF_ASSIST_SOCIAL', 'IN_PROF_BIBLIOTECARIO', 'IN_PROF_COORDENADOR', 'IN_PROF_FONAUDIOLOGO', 'IN_PROF_GESTAO', 'IN_PROF_MONITORES', 'IN_PROF_NUTRICIONISTA', 'IN_PROF_PEDAGOGIA', 'IN_PROF_PSICOLOGO', 'IN_PROF_REVISOR_BRAILLE', 'IN_PROF_SAUDE', 'IN_PROF_SECRETARIO', 'IN_PROF_SEGURANCA', 'IN_PROF_SERVICOS_GERAIS', 'IN_PROF_TEC', 'IN_PROF_TRAD_LIBRAS', 'QT_MAT_MED_CT', 'QT_MAT_MED_CT_1', 'QT_MAT_MED_CT_2', 'QT_MAT_MED_CT_3', 'QT_MAT_MED_CT_4', 'QT_MAT_MED_CT_NS']

Colunas em Censo 2025 mas não em Censo 2024:
Total: 497
['CO_REGIAO_ADMINISTRATIVA', 'IN_COMUM_CRECHE', 'IN_COMUM_EJA_FUND', 'IN_COMUM_EJA_MEDIO', 'IN_COMUM_EJA_PROF', 'IN_COMUM_FUND_AF', 'IN_COMUM_FUND_A

### Atenção

Parei o pipeline de ETL aqui. Foi definido o schema mas ainda não foi feito o merge final.

## Exploração

In [14]:
exploracao = df_censo_ac_2024#[df_censo_ac_2024["IN_LOCAL_FUNC_PREDIO_ESCOLAR"] == 0]

df_exploracao = exploracao[['NO_MUNICIPIO','NO_ENTIDADE', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'TP_OCUPACAO_PREDIO_ESCOLAR', 
                            'IN_LOCAL_FUNC_GALPAO', "IN_LOCAL_FUNC_SALAS_OUTRA_ESC", 
                            "IN_LOCAL_FUNC_OUTROS",]]

df_exploracao

,NO_MUNICIPIO,NO_ENTIDADE,IN_LOCAL_FUNC_PREDIO_ESCOLAR,TP_OCUPACAO_PREDIO_ESCOLAR,IN_LOCAL_FUNC_GALPAO,IN_LOCAL_FUNC_SALAS_OUTRA_ESC,IN_LOCAL_FUNC_OUTROS
1424,Acrelândia,ESC ALTINA MAGALHAES DA SILVA,1.0,1.0,0.0,0.0,1.0
1425,Acrelândia,ESC MARIA DE JESUS RIBEIRO,1.0,1.0,1.0,0.0,0.0
1426,Acrelândia,ESC PROF PEDRO DE CASTRO MEIRELES,1.0,1.0,0.0,0.0,0.0
1428,Acrelândia,ESC SANTA LUCIA III,1.0,1.0,0.0,1.0,0.0
1429,Acrelândia,ESC BRANCA DE NEVE,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
3078,Porto Acre,ESC PARAISO DO SABER,1.0,3.0,0.0,0.0,0.0
3079,Porto Acre,ESC JADER SARAIVA MACHADO,1.0,1.0,0.0,0.0,0.0
3080,Porto Acre,ESC SAO JOSE I,1.0,1.0,0.0,0.0,0.0
3081,Porto Acre,ESC RAIO DE LUZ,1.0,1.0,0.0,0.0,0.0


In [16]:
exploracao = df_censo_ac_2024[
    (df_censo_ac_2024["TP_LOCALIZACAO"] == 2) &
    (df_censo_ac_2024["QT_SALAS_UTILIZADAS_FORA"] != 0)
]

df_exploracao = exploracao[['NO_MUNICIPIO','NO_ENTIDADE', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'TP_OCUPACAO_PREDIO_ESCOLAR', 
                            'IN_LOCAL_FUNC_GALPAO', "IN_LOCAL_FUNC_SALAS_OUTRA_ESC", 
                            "IN_LOCAL_FUNC_OUTROS", 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA']]

print(df_exploracao.shape)

df_exploracao['NO_MUNICIPIO'].value_counts()



(268, 10)


NO_MUNICIPIO
Feijó                   53
Tarauacá                46
Santa Rosa do Purus     33
Sena Madureira          22
Rodrigues Alves         21
Brasiléia               12
Xapuri                  12
Assis Brasil            11
Cruzeiro do Sul          9
Jordão                   8
Marechal Thaumaturgo     7
Bujari                   5
Manoel Urbano            5
Capixaba                 4
Mâncio Lima              4
Porto Walter             4
Rio Branco               4
Plácido de Castro        3
Acrelândia               2
Porto Acre               2
Epitaciolândia           1
Name: count, dtype: int64

In [17]:
exploracao = df_censo_ac_2024[df_censo_ac_2024['IN_COZINHA'] == 0]

df_exploracao = exploracao[['NO_MUNICIPIO','NO_ENTIDADE', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'TP_OCUPACAO_PREDIO_ESCOLAR', 
                            'IN_LOCAL_FUNC_GALPAO', "IN_LOCAL_FUNC_SALAS_OUTRA_ESC", 
                            "IN_LOCAL_FUNC_OUTROS", 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 
                            'QT_SALAS_UTILIZADAS_FORA', 'IN_COZINHA']
]

#print(df_exploracao.value_counts())

print(df_exploracao.shape)

df_exploracao['IN_COZINHA'].value_counts()


(128, 11)


IN_COZINHA
0.0    128
Name: count, dtype: int64

In [18]:
exploracao = df_censo_ac_2024[['IN_COZINHA', 'TP_LOCALIZACAO']]

exploracao.value_counts()

IN_COZINHA  TP_LOCALIZACAO
1.0         2                 947
            1                 448
0.0         2                 117
            1                  11
Name: count, dtype: int64

In [19]:
exploracao = df_censo_ac_2024[['IN_COZINHA', 'IN_LOCAL_FUNC_GALPAO']]

exploracao.value_counts()

IN_COZINHA  IN_LOCAL_FUNC_GALPAO
1.0         0.0                     1371
0.0         0.0                      102
            1.0                       26
1.0         1.0                       24
Name: count, dtype: int64

In [20]:
exploracao = df_censo_ac_2024[['IN_COZINHA', 'IN_LOCAL_FUNC_OUTROS']]

exploracao.value_counts()

IN_COZINHA  IN_LOCAL_FUNC_OUTROS
1.0         0.0                     1277
            1.0                      118
0.0         0.0                      103
            1.0                       25
Name: count, dtype: int64

In [21]:

exploracao = df_censo_ac_2024['QT_PROF_ADMINISTRATIVOS']

exploracao.value_counts()

QT_PROF_ADMINISTRATIVOS
0.0     1030
1.0      202
2.0      134
3.0       70
4.0       39
6.0       19
5.0       14
9.0        5
7.0        4
11.0       3
8.0        3
Name: count, dtype: int64

In [22]:
exploracao = df_censo_ac_2024[['IN_PROF_SERVICOS_GERAIS', 'QT_PROF_SERVICOS_GERAIS',]]

exploracao.value_counts()

IN_PROF_SERVICOS_GERAIS  QT_PROF_SERVICOS_GERAIS
0.0                      0.0                        650
1.0                      1.0                        260
                         2.0                        113
                         5.0                         89
                         4.0                         88
                         6.0                         68
                         3.0                         59
                         8.0                         50
                         7.0                         49
                         10.0                        22
                         9.0                         22
                         11.0                        17
                         14.0                         9
                         13.0                         6
                         12.0                         5
                         16.0                         5
                         15.0                         5

In [23]:
exploracao = df_censo_ac_2024[['TP_LOCALIZACAO', 'IN_ORGAO_NENHUM']]

exploracao.value_counts()


TP_LOCALIZACAO  IN_ORGAO_NENHUM
2               1.0                579
                0.0                485
1               0.0                411
                1.0                 48
Name: count, dtype: int64

In [24]:
exploracao = df_censo_ac_2024[['TP_LOCALIZACAO', 'TP_AEE']]

exploracao.value_counts()


TP_LOCALIZACAO  TP_AEE
2               0.0       943
1               1.0       332
                0.0       124
2               1.0       121
1               2.0         3
Name: count, dtype: int64

In [25]:
exploracao = df_censo_ac_2024[['QT_MAT_MED', 'QT_MAT_MED_NM', 'QT_MAT_MED_PROP', 'QT_MAT_MED_CT']]

exploracao.sum()
 



QT_MAT_MED         40079.0
QT_MAT_MED_NM          0.0
QT_MAT_MED_PROP    38133.0
QT_MAT_MED_CT       1946.0
dtype: float64

In [ ]:
df_exploracao_prof = df_censo_ac_2025[df_censo_ac_2025['TP_LOCALIZACAO'] == 2]

exploracao = df_exploracao_prof[['QT_DOC_BAS', 'QT_DOC_BAS_DOCENTE', 'QT_DOC_BAS_AUXILIAR', 'QT_DOC_BAS_PROFI_MONITOR', 
                               'QT_DOC_BAS_TRADUTOR_LIBRAS', 'QT_DOC_BAS_TITULAR_EAD', 'QT_DOC_BAS_TUTOR_AUX_EAD',
                               'QT_DOC_BAS_GUIA_INTERPRETE', 'QT_DOC_BAS_APOIO_PCD', 'QT_DOC_BAS_INSTRUTOR_EP']]

exploracao.head(50)

In [ ]:
exploracao = df_censo_ac_2025[[ 'QT_DOC_BAS_ESPEC_CRE', 'QT_DOC_INF_CRE', 
                               'QT_DOC_BAS_ESPEC_PRE_ESCOLA', 'QT_DOC_INF_PRE', 
                               'QT_DOC_BAS_ESPEC_ANOS_INICIAIS', 'QT_DOC_FUND_AI', 
                               'QT_DOC_BAS_ESPEC_ANOS_FINAIS', 'QT_DOC_FUND_AF', 
                               'QT_DOC_BAS_ESPEC_ENS_MEDIO', 'QT_DOC_MED']]

exploracao.head(50)

In [33]:
# Caça anexos, turmas fora da sala
df_exploracao_turmas = df_censo_ac_2025[df_censo_ac_2025['TP_LOCALIZACAO'] == 2]

exploracao = df_exploracao_turmas[['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_LOCALIZACAO',
                                   'IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'TP_OCUPACAO_PREDIO_ESCOLAR', 'IN_LOCAL_FUNC_GALPAO', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 
                                   'IN_LOCAL_FUNC_OUTROS', 'IN_PREDIO_COMPARTILHADO', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA',
                                   'QT_MAT_BAS', 'QT_DOC_BAS', 'QT_DOC_FUND_AI_MULTIETAPA', 'QT_DOC_FUND_AF_MULTI', 'QT_DOC_MED_PROP_NS', 
                                   'QT_TUR_BAS', 'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N'
                                   ]
]

exploracao.head(50)

exploracao.to_csv("caca_anexo.csv", index=False)

In [25]:
df_exploracao_turmas = df_censo_ac_2025[df_censo_ac_2025['TP_LOCALIZACAO'] == 2]

exploracao = df_exploracao_turmas[['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_DEPENDENCIA',
                                   'QT_MAT_BAS', 'QT_TUR_BAS', 'QT_DOC_BAS', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 
                                   'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N'
                                   ]
]

exploracao.head(50)

exploracao.to_csv("caca_anexo_conciso.csv", index=False)

In [26]:
df_exploracao_turmas = df_censo_ac_2025[
    (df_censo_ac_2025['NO_MUNICIPIO'] == "Cruzeiro do Sul") &
    (df_censo_ac_2025['TP_DEPENDENCIA'] == 2) &
    (df_censo_ac_2025["TP_LOCALIZACAO"] == 2) &
    (df_censo_ac_2025["QT_TUR_BAS"] > 1) &
    (df_censo_ac_2025['QT_SALAS_UTILIZADAS'] > df_censo_ac_2025['QT_TUR_BAS']/2) &
    #(df_censo_ac_2025['QT_TUR_BAS_DM'] > df_censo_ac_2025['QT_TUR_BAS_D']/2)
    (
        (df_censo_ac_2025['QT_TUR_BAS_DM'] > df_censo_ac_2025['QT_TUR_BAS_D']/2) |
        (df_censo_ac_2025['QT_TUR_BAS_DV'] > df_censo_ac_2025['QT_TUR_BAS_D']/2)
    )
]

df_exploracao_turmas = df_exploracao_turmas[['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_DEPENDENCIA',
                      'QT_MAT_BAS', 'QT_TUR_BAS', 'QT_DOC_BAS', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 
                      'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N']
]

print(df_exploracao_turmas.shape)

df_exploracao_turmas.head(50) 

(20, 13)


,NO_MUNICIPIO,NO_ENTIDADE,TP_DEPENDENCIA,QT_MAT_BAS,QT_TUR_BAS,QT_DOC_BAS,QT_SALAS_UTILIZADAS,QT_SALAS_UTILIZADAS_DENTRO,QT_SALAS_UTILIZADAS_FORA,QT_TUR_BAS_D,QT_TUR_BAS_DM,QT_TUR_BAS_DV,QT_TUR_BAS_N
156,Cruzeiro do Sul,ESC 7 DE SETEMBRO,2,155,9,13,9.0,9.0,0.0,9,5,4,0
163,Cruzeiro do Sul,ESC AUGUSTO SEVERO,2,100,7,11,6.0,6.0,0.0,6,2,4,1
166,Cruzeiro do Sul,ESC CORA CORALINA,2,103,6,10,5.0,5.0,0.0,6,2,4,0
184,Cruzeiro do Sul,ESC PROF FRANCISCO ALBECIR BRITO DA SILVA,2,77,6,5,6.0,6.0,0.0,6,2,4,0
193,Cruzeiro do Sul,ESC MANOEL BRAZ DE MELO,2,392,14,27,8.0,8.0,0.0,14,6,8,0
194,Cruzeiro do Sul,ESC MARCILIO NUNES RIBEIRO II,2,112,5,11,4.0,4.0,0.0,5,3,2,0
196,Cruzeiro do Sul,ESC MARIA DE NAZARE SANTIAGO,2,230,10,13,8.0,8.0,0.0,10,6,4,0
199,Cruzeiro do Sul,ESC NORBERTO ASSUNCAO CAVALCANTE,2,134,9,11,6.0,6.0,0.0,9,5,4,0
209,Cruzeiro do Sul,ESC PLACIDO DE CASTRO,2,178,8,13,5.0,3.0,2.0,8,5,3,0
217,Cruzeiro do Sul,ESC RAINHA DA FLORESTA,2,154,12,10,12.0,8.0,4.0,12,7,5,0


In [63]:
colunas_interesse = ['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_DEPENDENCIA',
                     'QT_MAT_BAS', 'QT_TUR_BAS', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 
                     'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N']

df_1 = df_censo_ac_2025[
    (df_censo_ac_2025["TP_LOCALIZACAO"] == 2) &
    (df_censo_ac_2025["QT_TUR_BAS"] > 1) &
    (df_censo_ac_2025['QT_SALAS_UTILIZADAS'] > df_censo_ac_2025['QT_TUR_BAS']/2)
    ]


df_2 = df_censo_ac_2025[
    (df_censo_ac_2025["TP_LOCALIZACAO"] == 2) &
    (df_censo_ac_2025["QT_TUR_BAS"] > 1) &
    (df_censo_ac_2025['QT_SALAS_UTILIZADAS'] > df_censo_ac_2025['QT_TUR_BAS']/2) &
    (df_censo_ac_2025['QT_TUR_BAS_DM'] > df_censo_ac_2025['QT_TUR_BAS_D']/2)
    ]


df_3 = df_censo_ac_2025[
    (df_censo_ac_2025["TP_LOCALIZACAO"] == 2) &
    (df_censo_ac_2025["QT_TUR_BAS"] > 1) &
    (df_censo_ac_2025['QT_SALAS_UTILIZADAS'] > df_censo_ac_2025['QT_TUR_BAS']/2) &
    (
        (df_censo_ac_2025['QT_TUR_BAS_DM'] > df_censo_ac_2025['QT_TUR_BAS_D']/2) |
        (df_censo_ac_2025['QT_TUR_BAS_DV'] > df_censo_ac_2025['QT_TUR_BAS_D']/2)
    )
]
    

# linhas presentes em df_1 e ausentes em df_2

df_diff = df_1.merge(df_2, how="left", indicator=True)
df_diff = df_diff[df_diff["_merge"] == "left_only"]

df_diff = df_diff[['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_DEPENDENCIA',
                  'QT_MAT_BAS', 'QT_TUR_BAS', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 
                  'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N']
]

print(df_1.shape)
print(df_2.shape)
print(df_3.shape)

#print(df_diff.shape)
#df_diff.head(50)

#df_1.to_csv("anexo_amplo.csv", index=False)
#df_2.to_csv("anexo_rest_1.csv", index=False)



(530, 882)
(371, 882)
(457, 882)


In [19]:
colunas_interesse = ['NO_MUNICIPIO', 'NO_ENTIDADE', 'TP_DEPENDENCIA',
                     'QT_MAT_BAS', 'QT_TUR_BAS', 'QT_SALAS_UTILIZADAS', 'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 
                     'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N']

df_base = df_censo_ac_2025[
    (df_censo_ac_2025['TP_LOCALIZACAO'] == 2) &
    (df_censo_ac_2025['NO_MUNICIPIO'] == "Cruzeiro do Sul") &
    (df_censo_ac_2025['TP_DEPENDENCIA'] == 2)
]

df_1 = df_base[
    (df_base["QT_TUR_BAS"] > 1) &
    (df_base['QT_SALAS_UTILIZADAS'] > df_base['QT_TUR_BAS']/2)
    ]


df_2 = df_base[
    (df_base["QT_TUR_BAS"] > 1) &
    (df_base['QT_SALAS_UTILIZADAS'] > df_base['QT_TUR_BAS']/2) &
    (df_base['QT_TUR_BAS_DM'] > df_base['QT_TUR_BAS_D']/2)
    ]


df_3 = df_base[
    (df_base["TP_LOCALIZACAO"] == 2) &
    (df_base["QT_TUR_BAS"] > 1) &
    (df_base['QT_SALAS_UTILIZADAS'] > df_base['QT_TUR_BAS']/2) &
    (
        (df_base['QT_TUR_BAS_DM'] > df_base['QT_TUR_BAS_D']/2) |
        (df_base['QT_TUR_BAS_DV'] > df_base['QT_TUR_BAS_D']/2)
    )
]

df_4 =  df_base[
    (df_base["QT_TUR_BAS"] >= 4) &
    ((df_base["QT_TUR_BAS"] / df_base["QT_SALAS_UTILIZADAS"]) >= 4)
    #(df_base['QT_SALAS_UTILIZADAS'] > df_base['QT_TUR_BAS']/2)
    ]

#df_1[colunas_interesse].to_csv("anexo_czs_estadual.csv", index=False)

print(len(df_1) - len(df_2))

df_1[colunas_interesse]

13


,NO_MUNICIPIO,NO_ENTIDADE,TP_DEPENDENCIA,QT_MAT_BAS,QT_TUR_BAS,QT_SALAS_UTILIZADAS,QT_SALAS_UTILIZADAS_DENTRO,QT_SALAS_UTILIZADAS_FORA,QT_TUR_BAS_D,QT_TUR_BAS_DM,QT_TUR_BAS_DV,QT_TUR_BAS_N
156,Cruzeiro do Sul,ESC 7 DE SETEMBRO,2,155,9,9.0,9.0,0.0,9,5,4,0
163,Cruzeiro do Sul,ESC AUGUSTO SEVERO,2,100,7,6.0,6.0,0.0,6,2,4,1
166,Cruzeiro do Sul,ESC CORA CORALINA,2,103,6,5.0,5.0,0.0,6,2,4,0
168,Cruzeiro do Sul,ESC DOM PEDRO II,2,40,4,4.0,4.0,0.0,4,2,2,0
181,Cruzeiro do Sul,ESC HUMBERTO DE CAMPOS,2,133,12,10.0,7.0,3.0,12,6,6,0
184,Cruzeiro do Sul,ESC PROF FRANCISCO ALBECIR BRITO DA SILVA,2,77,6,6.0,6.0,0.0,6,2,4,0
191,Cruzeiro do Sul,ESC JUAREZ IBERNON,2,316,10,6.0,6.0,0.0,10,5,5,0
193,Cruzeiro do Sul,ESC MANOEL BRAZ DE MELO,2,392,14,8.0,8.0,0.0,14,6,8,0
194,Cruzeiro do Sul,ESC MARCILIO NUNES RIBEIRO II,2,112,5,4.0,4.0,0.0,5,3,2,0
196,Cruzeiro do Sul,ESC MARIA DE NAZARE SANTIAGO,2,230,10,8.0,8.0,0.0,10,6,4,0


In [ ]:
SCHEMA_CURICA = [

    # ESCOLAS
    # identificação
    'NU_ANO_CENSO',
    'SG_UF',
    'NO_MUNICIPIO',
    'CO_MUNICIPIO',
    'CO_ENTIDADE',
    'NO_ENTIDADE',
    
    # caracterização
    'TP_DEPENDENCIA',
    'TP_LOCALIZACAO',
    'TP_LOCALIZACAO_DIFERENCIADA',
    'DS_ENDERECO',
    'NU_ENDERECO',
    'DS_COMPLEMENTO',
    'NO_BAIRRO',
    'CO_CEP',
    'TP_SITUACAO_FUNCIONAMENTO',
    'LATITUDE',
    'LONGITUDE',
    
    # infra prédio
    'IN_LOCAL_FUNC_PREDIO_ESCOLAR',
    'TP_OCUPACAO_PREDIO_ESCOLAR',
    'IN_LOCAL_FUNC_GALPAO',
    'IN_LOCAL_FUNC_SALAS_OUTRA_ESC',
    'IN_LOCAL_FUNC_OUTROS',
    'IN_PREDIO_COMPARTILHADO',

    # infra água
    'IN_AGUA_POTAVEL',
    'IN_AGUA_REDE_PUBLICA',
    'IN_AGUA_POCO_ARTESIANO',
    'IN_AGUA_CACIMBA',
    'IN_AGUA_FONTE_RIO',
    'IN_AGUA_INEXISTENTE',
    'IN_AGUA_CARRO_PIPA',
    'IN_ENERGIA_REDE_PUBLICA',

    # infra energia elétrica
    'IN_ENERGIA_REDE_PUBLICA',
    'IN_ENERGIA_GERADOR_FOSSIL',
    'IN_ENERGIA_RENOVAVEL',
    'IN_ENERGIA_INEXISTENTE',

    # infra esgoto
    'IN_ESGOTO_REDE_PUBLICA',
    'IN_ESGOTO_FOSSA_SEPTICA',
    'IN_ESGOTO_FOSSA_COMUM',
    'IN_ESGOTO_FOSSA',
    'IN_ESGOTO_INEXISTENTE',

    # infra dependências físicas
    'IN_ALMOXARIFADO',
    'IN_BANHEIRO', # infra prédio
    'IN_BANHEIRO_FUNCIONARIOS',
    'IN_BIBLIOTECA',
    'IN_BIBLIOTECA_SALA_LEITURA',
    'IN_COZINHA', # alimentacao escolar
    'IN_DESPENSA', # alimentacao escolar
    'IN_LABORATORIO_CIENCIAS',
    'IN_LABORATORIO_INFORMATICA',
    'IN_REFEITORIO',
    'IN_SALA_DIRETORIA',
    'IN_SALA_LEITURA',
    'IN_SALA_PROFESSOR',
    'IN_SECRETARIA',
    'IN_SALA_ATENDIMENTO_ESPECIAL',
    
    # salas utilizadas
    'QT_SALAS_UTILIZADAS_DENTRO',
    'QT_SALAS_UTILIZADAS_FORA',
    'QT_SALAS_UTILIZADAS',

    # equipamentos de informática e multimídia
    'IN_EQUIP_PARABOLICA',
    'IN_COMPUTADOR',
    'IN_EQUIP_COPIADORA',
    'IN_EQUIP_IMPRESSORA',
    'IN_EQUIP_IMPRESSORA_MULT',
    'IN_EQUIP_SCANNER',
    'IN_EQUIP_NENHUM',
    'IN_EQUIP_DVD',
    'QT_EQUIP_DVD',
    'IN_EQUIP_SOM',
    'QT_EQUIP_SOM',
    'IN_EQUIP_TV',
    'QT_EQUIP_TV',
    'IN_EQUIP_LOUSA_DIGITAL',
    'QT_EQUIP_LOUSA_DIGITAL',
    'IN_EQUIP_MULTIMIDIA',
    'QT_EQUIP_MULTIMIDIA',
    'IN_DESKTOP_ALUNO',
    'QT_DESKTOP_ALUNO',
    'IN_COMP_PORTATIL_ALUNO',
    'QT_COMP_PORTATIL_ALUNO',
    'IN_TABLET_ALUNO',
    'QT_TABLET_ALUNO',
    'IN_INTERNET', # disponibilidade de internet
    'IN_INTERNET_ALUNOS',
    'IN_INTERNET_ADMINISTRATIVO',
    'IN_INTERNET_APRENDIZAGEM',
    'IN_INTERNET_COMUNIDADE',
    'IN_ACESSO_INTERNET_COMPUTADOR',
    'IN_ACES_INTERNET_DISP_PESSOAIS',
    'TP_REDE_LOCAL',
    'IN_BANDA_LARGA',

    # recursos humanos
    'IN_PROF_ADMINISTRATIVOS',
    'QT_PROF_ADMINISTATIVOS',
    'IN_PROF_SERVICOS_GERAIS',
    'QT_PROF_SERVICOS_GERAIS',
    'QT_PROF_ALIMENTACAO'
    'QT_PROF_SECRETARIO',
    'QT_PROF_SEGURANCA',
    'QT_PROF_MONITORES',
    'QT_PROF_GESTAO',
    
    # alimentação escolar
    'IN_ALIMENTACAO',

    # órgãos colegiados
    'IN_ORGAO_ASS_PAIS',
    'IN_ORGAO_ASS_PAIS_MESTRES',
    'IN_ORGAO_CONSELHO_ESCOLAR',
    'IN_ORGAO_GREMIO_ESTUDANTIL',
    'IN_ORGAO_OUTROS',
    'IN_ORGAO_NENHUM',

    # ensino especial
    'TP_AEE',
    'IN_COMUM_CRECHE',
    'IN_COMUM_PRE',
    'IN_COMUM_FUND_AI',
    'IN_COMUM_FUND_AF',
    'IN_COMUM_MEDIO_MEDIO',
    'IN_COMUM_MEDIO_INTEGRADO',
    'IN_COMUM_MEDIO_FIC',
    'IN_COMUM_MEDIO_NORMAL',
    'IN_ESP_EXCLUSIVA_CRECHE',
    'IN_ESP_EXCLUSIVA_PRE',
    'IN_ESP_EXCLUSIVA_FUND_AI',
    'IN_ESP_EXCLUSIVA_FUND_AF',
    'IN_ESP_EXCLUSIVA_MEDIO_MEDIO',
    'IN_ESP_EXCLUSIVA_MEDIO_INTEGR',
    'IN_ESP_EXCLUSIVA_MEDIO_FIC',
    'IN_ESP_EXCLUSIVA_MEDIO_NORMAL',

    # modalidade de ensino
    'IN_ESCOLARIZACAO', # qualquer etapa disponível com matrícula
    'IN_MEDIACAO_PRESENCIAL',
    'IN_MEDIACAO_SEMIPRESENCIAL',
    'IN_MEDIACAO_EAD',
    'IN_ESPECIAL_EXCLUSIVA',
    'IN_REGULAR',


    # MATRÍCULAS
    # quantidade de matrículas
    'QT_MAT_BAS', # total geral
    'QT_MAT_INF',
    'QT_MAT_INF_CRE',
    'QT_MAT_INF_PRE',
    'QT_MAT_FUND',
    'QT_MAT_FUND_AI',
    'QT_MAT_FUND_AI_1',
    'QT_MAT_FUND_AI_2',
    'QT_MAT_FUND_AI_3',
    'QT_MAT_FUND_AI_4',
    'QT_MAT_FUND_AI_5',
    'QT_MAT_FUND_AF',
    'QT_MAT_FUND_AF_6',
    'QT_MAT_FUND_AF_7',
    'QT_MAT_FUND_AF_8',
    'QT_MAT_FUND_AF_9',
    'QT_MAT_MED', # somatório de todas as modalidades
    'QT_MAT_MED_PROP', # é o tradicional
    'QT_MAT_MED_PROP_1', # 1o. ano
    'QT_MAT_MED_PROP_2',
    'QT_MAT_MED_PROP_3',
    'QT_MAT_MED_PROP_4',
    'QT_MAT_MED_PROP_NS', # não seriado!
    'QT_MAT_MED_IFTP_CT', # curso técnico
    'QT_MAT_MED_IFTP_QP', # itinerário formativo técnico
    'QT_MAT_MED_IFA', # itinerário formativo aprofundamento
    'QT_MAT_MED_NM', # magistério

    # matrículas ensino integral 
    'QT_MAT_INF_INT',
    'QT_MAT_INF_CRE_INT',
    'QT_MAT_INF_PRE_INT',
    'QT_MAT_FUND_INT',
    'QT_MAT_FUND_AI_INT',
    'QT_MAT_FUND_AF_INT',
    'QT_MAT_MED_INT',
    
    # matrículas educação especial
    'QT_MAT_ESP', 
    'QT_MAT_ESP_INF', # detalhamento a partir de 2025
    'QT_MAT_ESP_INF_CRE',
    'QT_MAT_ESP_INF_PRE',
    'QT_MAT_ESP_FUND',
    'QT_MAT_ESP_FUND_AI',
    'QT_MAT_ESP_FUND_AF',
    'QT_MAT_ESP_MED',

    # matrículas por turno, a partir de 2025
    'QT_MAT_INF_CRE_DM', # matutino
    'QT_MAT_INF_CRE_DV', # vespertino
    'QT_MAT_INF_CRE_N', # noturno
    'QT_MAT_INF_PRE_D',
    'QT_MAT_INF_PRE_DM',
    'QT_MAT_INF_PRE_DV',
    'QT_MAT_INF_PRE_N',
    'QT_MAT_FUND_D',
    'QT_MAT_FUND_DM',
    'QT_MAT_FUND_DV',
    'QT_MAT_FUND_N',
    'QT_MAT_FUND_AI_D',
    'QT_MAT_FUND_AI_DM',
    'QT_MAT_FUND_AI_DV',
    'QT_MAT_FUND_AI_N',
    'QT_MAT_FUND_AF_D',
    'QT_MAT_FUND_AF_DM',
    'QT_MAT_FUND_AF_DV',
    'QT_MAT_FUND_AF_N',
    'QT_MAT_MED_D',
    'QT_MAT_MED_DM',
    'QT_MAT_MED_DV',
    'QT_MAT_MED_N',

    # matrículas tempo integral
    'QT_MAT_INF_INT',
    'QT_MAT_INF_CRE_INT',
    'QT_MAT_INF_PRE_INT',
    'QT_MAT_FUND_INT',
    'QT_MAT_FUND_AI_INT',
    'QT_MAT_FUND_AF_INT',
    'QT_MAT_MED_INT',
    'QT_MAT_BAS_INT', # total, a partir de 2025

    # transporte escolar
    'QT_TRANSP_PUBLICO',
    'QT_TRANSP_RESP_EST',
    'QT_TRANSP_RESP_MUN',

    
    # DOCENTES
    # quantidade de docentes por etapa
    'QT_DOC_BAS',
    'QT_DOC_INF',
    'QT_DOC_INF_CRE',
    'QT_DOC_INF_PRE',
    'QT_DOC_FUND',
    'QT_DOC_FUND_AI',
    'QT_DOC_FUND_AI_1',
    'QT_DOC_FUND_AI_2',
    'QT_DOC_FUND_AI_3',
    'QT_DOC_FUND_AI_4',
    'QT_DOC_FUND_AI_5',
    'QT_DOC_FUND_AI_MULTIETAPA',
    'QT_DOC_FUND_AF',
    'QT_DOC_FUND_AF_6',
    'QT_DOC_FUND_AF_7',
    'QT_DOC_FUND_AF_8',
    'QT_DOC_FUND_AF_9',
    'QT_DOC_FUND_AF_MULTI',
    'QT_DOC_FUND_AF_CORRFLUXO',
    'QT_DOC_MED',
    'QT_DOC_MED_PROP',
    'QT_DOC_MED_PROP_1',
    'QT_DOC_MED_PROP_2',
    'QT_DOC_MED_PROP_3',
    'QT_DOC_MED_PROP_4',
    'QT_DOC_MED_PROP_NS', # Não seridado!
    'QT_DOC_ESP', # ensino especial
    'QT_DOC_ESP_CC', # ensino especial classes comuns
    'QT_DOC_ESP_CE', # ensino especial classes exclusivas
    
    # escolaridade dos docentes
    'QT_DOC_BAS_ESCO_EF',
    'QT_DOC_BAS_ESCO_EM',
    'QT_DOC_BAS_ESCO_SUP_GRAD',
    'QT_DOC_BAS_ESCO_SUP_GRAD_LICEN',
    'QT_DOC_BAS_ESCO_SUP_GRAD_SLICEN',
    'QT_DOC_BAS_ESCO_SUP_POS_ESPEC',
    'QT_DOC_BAS_ESCO_SUP_POS_MESTRA',
    'QT_DOC_BAS_ESCO_SUP_POS_DOUTO',
    'QT_DOC_BAS_ESCO_SUP_POS_NENHUM',
    
    # vínculo de trabalho dos docentes
    'QT_DOC_BAS_VINCULO_CONCUR',
    'QT_DOC_BAS_VINCULO_CONTRA',
    'QT_DOC_BAS_VINCULO_TERCEIR',
    'QT_DOC_BAS_VINCULO_CLT',

    # quantidade de docentes com cursos de formação continuada
    'QT_DOC_BAS_ESPEC_CRE',
    'QT_DOC_BAS_ESPEC_PRE_ESCOLA',
    'QT_DOC_BAS_ESPEC_ANOS_INICIAIS',
    'QT_DOC_BAS_ESPEC_ANOS_FINAIS',
    'QT_DOC_BAS_ESPEC_ENS_MEDIO',
    'QT_DOC_BAS_ESPEC_ED_ESPECIAL',

    # TURMAS
    # Quantidade de turmas
    'QT_TUR_BAS',
    'QT_TUR_INF',
    'QT_TUR_INF_CRE',
    'QT_TUR_INF_PRE',
    'QT_TUR_FUND',
    'QT_TUR_FUND_AI',
    'QT_TUR_FUND_AI_1',
    'QT_TUR_FUND_AI_2',
    'QT_TUR_FUND_AI_3',
    'QT_TUR_FUND_AI_4',
    'QT_TUR_FUND_AI_5',
    'QT_TUR_FUND_AI_MULTIETAPA', # multietapa
    'QT_TUR_FUND_AF',
    'QT_TUR_FUND_AF_6',
    'QT_TUR_FUND_AF_7',
    'QT_TUR_FUND_AF_8',
    'QT_TUR_FUND_AF_9',
    'QT_TUR_FUND_AF_MULTI', # multietapa
    'QT_TUR_FUND_AF_CORRFLUXO',
    'QT_TUR_MED',
    'QT_TUR_MED_PROP',
    'QT_TUR_MED_PROP_1',
    'QT_TUR_MED_PROP_2',
    'QT_TUR_MED_PROP_3',
    'QT_TUR_MED_PROP_4',
    'QT_TUR_MED_PROP_NS', # não seriado

    # turnos das turmas
    'QT_TUR_BAS_D',
    'QT_TUR_BAS_DM',
    'QT_TUR_BAS_DV',
    'QT_TUR_BAS_N',
    'QT_TUR_INF_CRE_D',
    'QT_TUR_INF_CRE_DM',
    'QT_TUR_INF_CRE_DV',
    'QT_TUR_INF_CRE_N',
    'QT_TUR_INF_PRE_D',
    'QT_TUR_INF_PRE_DM',
    'QT_TUR_INF_PRE_DV',
    'QT_TUR_INF_PRE_N',
    'QT_TUR_FUND_D',
    'QT_TUR_FUND_DM',
    'QT_TUR_FUND_DV',
    'QT_TUR_FUND_N',
    'QT_TUR_FUND_AI_D',
    'QT_TUR_FUND_AI_DM',
    'QT_TUR_FUND_AI_DV',
    'QT_TUR_FUND_AI_N',
    'QT_TUR_FUND_AF_D',
    'QT_TUR_FUND_AF_DM',
    'QT_TUR_FUND_AF_DV',
    'QT_TUR_FUND_AF_N',
    'QT_TUR_MED_D',
    'QT_TUR_MED_DM',
    'QT_TUR_MED_DV',
    'QT_TUR_MED_N',

    # ensino integral
    'QT_TUR_BAS_INT',
    'QT_TUR_INF_INT',
    'QT_TUR_INF_CRE_INT',
    'QT_TUR_INF_PRE_INT',
    'QT_TUR_FUND_INT',
    'QT_TUR_FUND_AI_INT',
    'QT_TUR_FUND_AF_INT',
    'QT_TUR_MED_INT',

    # ensino especial
    'QT_TUR_ESP',
    'QT_TUR_ESP_CC',
    'QT_TUR_ESP_CE',
    # turnos especial
    'QT_TUR_ESP_D',
    'QT_TUR_ESP_DM',
    'QT_TUR_ESP_DV',
    'QT_TUR_ESP_N',
    # especial integral
    'QT_TUR_ESP_INT',
    'QT_TUR_ESP_CC_INT',
    'QT_TUR_ESP_CE_INT',
]